# From tracking points to behavior: a pandas and NumPy walkthrough

**Question:** How can we turn tracked mouse positions into an interpretable description of movement?

This is a guided example of my analysis workflow: **load -> inspect -> filter -> interpolate -> measure -> check**.
We will focus on three ideas: selecting labeled data with pandas, masking unreliable observations,
and calculating movement with NumPy. Ask questions throughout!

| Minutes | Topic |
|---|---|
| 0-5 | The biological question and a short tracking video |
| 5-15 | Load a table and select one mouse / body part |
| 15-27 | Confidence scores, Boolean masks, and visual quality checks |
| 27-35 | Fill short gaps, leave long gaps unknown |
| 35-47 | Position, movement, and one summary plot |
| 47-60 | Questions; optional advanced examples |

**Presenter preparation:** Select the real data below, confirm the video frame rate, and run the notebook
from a fresh kernel before the meeting. Prepare a short original / filtered / interpolated video comparison
using the optional appendix. Skip helper internals during the main walkthrough.

The default is an explicitly **synthetic practice dataset**, so this notebook runs without the lab drive.
Its plots are demonstrations, not experimental findings. Dependencies: NumPy, pandas, and Matplotlib;
real HDF files additionally require PyTables, and optional video export requires OpenCV.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_DEMO_DATA = True  # Set False for your DeepLabCut recording.
FPS = 30.0           # Confirm against the actual recording before interpreting time.
CONFIDENCE_THRESHOLD = 0.6
MAX_GAP_FRAMES = 5   # Illustrative: justify the duration for your behavior and frame rate.
CLIP_START = 450
CLIP_STOP = 1260     # Exclusive everywhere in this notebook.
OUTPUT_DIR = Path.cwd() / "codersclub_outputs"
PLOT_COLORS = ["#b34cc2", "#c08a00", "#008caa"]
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False,
                     "axes.spines.right": False})


In [ ]:
DATA_DIR = Path( r"Z:\n2023_odor_related_behavior\other\Vorträge\261002_Coders_Club")
DATA_FILE = DATA_DIR / "multi_animal_example.h5"
VIDEO_FILE = DATA_DIR / "multi_animal_example.avi"


## 1. Load and inspect (5-15 min)

**Presenter:** Start with the prepared video if available. What would you want to measure from it?

The synthetic-data helper below is setup code; collapse it during the meeting.


In [ ]:
def make_demo_data(n_frames=1500):
    """Reproducible toy tracking data with low-confidence points and missing gaps."""
    rng = np.random.default_rng(42)
    columns = pd.MultiIndex.from_product(
        [["demo_tracker"], ["mouse1", "mouse2", "mouse3"],
         ["nose", "shoulder_left", "shoulder_right", "tail_base"],
         ["x", "y", "likelihood"]],
        names=["scorer", "individuals", "bodyparts", "coords"])
    result = pd.DataFrame(index=pd.RangeIndex(n_frames, name="frame"), columns=columns, dtype=float)
    t = np.arange(n_frames) / FPS
    offsets = {"nose": (12, 0), "shoulder_left": (0, -5),
               "shoulder_right": (0, 5), "tail_base": (-12, 0)}
    for i, animal in enumerate(["mouse1", "mouse2", "mouse3"]):
        phase = t * (0.18 + i * 0.03)
        phase[450:600] = phase[450]  # A stationary interval with tracking noise.
        for part, (dx, dy) in offsets.items():
            x = 220 + 65 * np.cos(phase + i) + dx + rng.normal(0, 0.4, n_frames)
            y = 180 + 50 * np.sin(phase + i) + dy + rng.normal(0, 0.4, n_frames)
            confidence = rng.uniform(0.75, 1, n_frames)
            bad = rng.random(n_frames) < 0.03
            bad[700:703] = True
            bad[850:885] = True
            confidence[bad] = 0.2
            x[bad] += rng.normal(0, 30, bad.sum())
            y[bad] += rng.normal(0, 30, bad.sum())
            for coord, values in [("x", x), ("y", y), ("likelihood", confidence)]:
                result.loc[:, ("demo_tracker", animal, part, coord)] = values
    return result

if USE_DEMO_DATA:
    raw_df = make_demo_data()
    print("SYNTHETIC PRACTICE DATA: not experimental results")
else:
    raw_df = pd.read_hdf(DATA_FILE)
    print(f"Loaded experimental data: {DATA_FILE}")

# Normalize single-animal files to the same structure as multi-animal files.
if "individuals" not in raw_df.columns.names:
    raw_df = pd.concat({"mouse1": raw_df}, axis=1, names=["individuals"])
    raw_df = raw_df.reorder_levels(["scorer", "individuals", "bodyparts", "coords"], axis=1)

scorers = raw_df.columns.get_level_values("scorer").unique()
if len(scorers) != 1:
    raise ValueError("Choose one scorer explicitly before continuing.")
df = raw_df.xs(scorers[0], level="scorer", axis=1).copy()
# The following calculations assume consecutive video frames in recording order.
if not np.array_equal(df.index.to_numpy(), np.arange(len(df))):
    raise ValueError("Check frame alignment: expected consecutive frame numbers starting at zero.")
animals = df.columns.get_level_values("individuals").unique().tolist()
bodyparts = df.columns.get_level_values("bodyparts").unique().tolist()
animal = animals[0]
part = "nose" if "nose" in bodyparts else bodyparts[0]
clip_start = min(CLIP_START, max(0, len(df) - 1))
clip_stop = min(CLIP_STOP, len(df))
print(f"{len(df):,} frames; {len(animals)} animals; {len(bodyparts)} body parts")


One **row is one frame**. A **column identifies an animal, a body part, and a measurement**.
`x` and `y` are image coordinates in pixels. `likelihood` is the tracker's confidence score.
The scorer level identifies the model; we selected one model above to simplify the table.

Start small: here are three columns for one body part of one animal.


In [ ]:
point = df.loc[:, (animal, part)].copy()
point.head()


In [ ]:
print("Column levels:", df.columns.names)
print("Animals:", animals)
print("Body parts:", bodyparts)
# .loc selects rows first, then labeled columns. ':' means all rows.
nose_x = df.loc[:, (animal, part, "x")]
print(nose_x.head())


**Pause:** What would you change to select a different animal or its y-coordinate?

A DataFrame is a labeled table; selecting one column gives a Series. NumPy arrays hold numerical
values without these row and column labels. Keep the labels until numerical array operations help.


## 2. Inspect confidence and mask unreliable points (15-27 min)

Confidence is useful for screening, but **0.6 does not establish a 60% probability that a point is correct**.
Check tracking visually when choosing a threshold. High-confidence mistakes can remain.


In [ ]:
likelihoods = df.xs("likelihood", level="coords", axis=1)
all_scores = likelihoods.to_numpy().ravel()  # Table -> array -> one dimension.
plt.hist(all_scores[np.isfinite(all_scores)], bins=np.linspace(0, 1, 21))
plt.axvline(CONFIDENCE_THRESHOLD, color="red", label="Example threshold")
plt.xlabel("Tracking confidence score")
plt.ylabel("Keypoint observations (all animals and frames)")
plt.legend()
plt.show()


In [ ]:
# First understand the operation on ONE point.
bad = point["likelihood"].isna() | (point["likelihood"] <= CONFIDENCE_THRESHOLD)
clean_point = point.copy()
clean_point.loc[bad, ["x", "y"]] = np.nan
example_rows = pd.DataFrame({
    "confidence": point["likelihood"], "mask": bad,
    "x_before": point["x"], "x_after": clean_point["x"]})
example_rows.head(15)


`True` in the mask means "replace these coordinates with missing values." `NaN` means unknown,
not zero: the animal has not moved to the corner of the image!

Now apply the same operation to every animal / body-part pair, preserving the original table.


In [ ]:
filtered_df = df.copy()
for individual, bodypart in likelihoods.columns:
    confidence = df.loc[:, (individual, bodypart, "likelihood")]
    bad = confidence.isna() | (confidence <= CONFIDENCE_THRESHOLD)
    filtered_df.loc[bad, (individual, bodypart, ["x", "y"])] = np.nan

clip_scores = likelihoods.iloc[clip_start:clip_stop]
flagged = clip_scores.isna() | (clip_scores <= CONFIDENCE_THRESHOLD)
print(f"Flagged {flagged.to_numpy().sum():,} / {flagged.size:,} keypoint observations "
      f"({100 * flagged.to_numpy().mean():.1f}%) in the selected clip.")


**Visual checkpoint:** Show the prepared original and filtered clips with identical frame ranges.
Which obvious mistakes disappeared? Which remained? Filtering removes observations; it does not repair tracking.


## 3. Fill only short internal gaps (27-35 min)

Interpolation estimates a path between two observed positions. It does not recover the true movement.
Only fill short gaps bounded by observations; keep longer gaps and unbounded ends unknown.
The acceptable gap depends on frame rate and the behavior being measured.

The helper's implementation is an optional advanced pandas example. For the main session, focus on its input and output.


In [ ]:
def interpolate_short_gaps(values, max_gap):
    """Fill complete internal missing runs only if their length is <= max_gap."""
    missing = values.isna()
    groups = missing.ne(missing.shift()).cumsum()
    lengths = missing.groupby(groups).transform("sum")
    candidate = values.interpolate(method="linear", limit_area="inside")
    return candidate.mask(missing & (lengths > max_gap))

example = pd.Series([0., 1., np.nan, 3., 4., np.nan, np.nan, np.nan, 8., np.nan])
filled = interpolate_short_gaps(example, max_gap=2)
plt.plot(example, "o", label="Observed")
plt.plot(filled, ".--", label="After short-gap interpolation")
plt.xlabel("Frame")
plt.ylabel("Example x-position (pixels)")
plt.legend()
plt.show()


In [ ]:
interpolated_df = filtered_df.copy()
coordinate_cols = filtered_df.columns.get_level_values("coords").isin(["x", "y"])
for col in filtered_df.columns[coordinate_cols]:
    interpolated_df[col] = interpolate_short_gaps(filtered_df[col], MAX_GAP_FRAMES)

before = filtered_df.loc[:, coordinate_cols].iloc[clip_start:clip_stop]
after = interpolated_df.loc[:, coordinate_cols].iloc[clip_start:clip_stop]
print(f"Filled {(before.isna() & after.notna()).to_numpy().sum():,} coordinate values "
      "(x and y counted separately).")
print(f"Maximum filled gap: {MAX_GAP_FRAMES / FPS:.3f} seconds")


**Pause:** Would linear interpolation be reasonable across a rapid turn? Across a long occlusion?
Show the prepared interpolated clip if available.

**Advanced explanation:** `shift` compares adjacent missingness flags; `cumsum` numbers consecutive runs;
`groupby(...).transform('sum')` broadcasts each missing run's length back to its rows.


## 4. From coordinates to movement (35-47 min)

Estimate position from a fixed set of body landmarks. For this demonstration we use both shoulders
and the tail base, requiring all three to be available. This avoids changing the set being averaged
from frame to frame. It is still a **position proxy**, affected by posture and tracking errors.
Choose landmarks appropriate for your recording; this is not a validated center-of-mass estimate.


In [ ]:
CENTER_PARTS = ["shoulder_left", "shoulder_right", "tail_base"]
missing_parts = set(CENTER_PARTS) - set(bodyparts)
if missing_parts:
    raise ValueError(f"Choose available CENTER_PARTS; unavailable: {missing_parts}")

centers = {}
for individual in animals:
    x = interpolated_df.loc[:, (individual, CENTER_PARTS, "x")].to_numpy()
    y = interpolated_df.loc[:, (individual, CENTER_PARTS, "y")].to_numpy()
    valid = np.isfinite(x).all(axis=1) & np.isfinite(y).all(axis=1)
    center = pd.DataFrame(index=df.index, columns=["x", "y"], dtype=float)
    center.loc[valid, "x"] = x[valid].mean(axis=1)  # Across body parts, within each frame.
    center.loc[valid, "y"] = y[valid].mean(axis=1)
    centers[individual] = center

fig, ax = plt.subplots()
for individual, color in zip(animals, [PLOT_COLORS[i % len(PLOT_COLORS)] for i in range(len(animals))]):
    positions = centers[individual].iloc[clip_start:clip_stop]
    ax.plot(positions["x"], positions["y"], color=color, label=individual)
ax.invert_yaxis()  # Image y-coordinates increase downward.
ax.set(xlabel="x (pixels)", ylabel="y (pixels)", title="Position proxy in the selected clip")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()


NumPy highlight: **changes in x and y -> Pythagoras -> distance between consecutive frames**.
`np.diff` produces one fewer value; we add a missing first value because there is no preceding frame.
Missing positions make adjacent steps unknown. We do not bridge those gaps.

Step distance is in **pixels**; multiplying by frames/second gives **pixels/second**.
Converting pixels to centimeters would require a spatial calibration.


In [ ]:
x = centers[animal]["x"].to_numpy()
y = centers[animal]["y"].to_numpy()
step_px = np.r_[np.nan, np.hypot(np.diff(x), np.diff(y))]
speed_px_s = step_px * FPS
pd.DataFrame({"step_distance_px": step_px, "speed_px_s": speed_px_s}).head()


In [ ]:
# Organize derived quantities separately from the source tracking data.
per_animal = {}
for individual in animals:
    center = centers[individual]
    step = np.r_[np.nan, np.hypot(np.diff(center["x"]), np.diff(center["y"]))]
    per_animal[individual] = center.rename(columns={"x": "center_x", "y": "center_y"}).assign(
        step_px=step,
        speed_px_s=step * FPS,
        observed_cumulative_distance_px=np.nancumsum(step))
metrics = pd.concat(per_animal, axis=1, names=["individual", "metric"])
metrics.index.name = "frame"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, individual in enumerate(animals):
    values = metrics[individual]
    color = PLOT_COLORS[i % len(PLOT_COLORS)]
    clip = values.iloc[clip_start:clip_stop]
    axes[0].plot((clip.index - clip_start) / FPS, clip["speed_px_s"], color=color, label=individual)
    axes[1].plot(values.index / FPS, values["observed_cumulative_distance_px"], color=color, label=individual)
axes[0].set(xlabel="Time from clip start (s)", ylabel="Speed (pixels/s)", title="Frame-to-frame movement")
axes[1].set(xlabel="Recording time (s)", ylabel="Observed cumulative distance (pixels)",
            title="Sum of available steps; gaps omitted")
axes[1].legend()
plt.tight_layout()
plt.show()

summary = pd.DataFrame({individual: {
    "observed_distance_px": metrics[individual]["step_px"].sum(min_count=1),
    "valid_step_fraction": metrics[individual]["step_px"].iloc[1:].notna().mean()
} for individual in animals}).T
summary


**Interpretation checkpoint:** Flat cumulative sections can mean missing tracking, not just rest.
The distance omits unknown intervals, includes estimates across interpolated gaps, and can be inflated by jitter.
Report coverage alongside the result. A stationary mouse can have nonzero apparent movement.

**Stop here for the main walkthrough.** Questions: Which decisions most affect the result? How would we
validate the position proxy and confidence threshold? What would change with a different camera frame rate?


## Optional: smoothing and a candidate immobility rule

A rolling mean makes a trace easier to view but does not remove the positive speed baseline caused by jitter.
Require a complete window and keep the raw speed. Unlike zero-padded convolution, this leaves edges unknown.

Do **not** subtract each recording's median and call the resulting zeros immobility: that would make
at least about half the valid samples zero by construction. Instead, choose a fixed threshold from
manually checked stationary/moving clips and validate on separate clips. The value below is only illustrative.


In [ ]:
WINDOW_FRAMES = 15
IMMOBILITY_THRESHOLD_PX_S = 15.0  # Demonstration only; calibrate and validate before scientific use.
selected_speed = metrics[animal]["speed_px_s"]
smoothed_speed = selected_speed.rolling(WINDOW_FRAMES, center=True, min_periods=WINDOW_FRAMES).mean()
immobile = pd.Series(pd.NA, index=df.index, dtype="boolean", name="candidate_immobile")
known = smoothed_speed.notna()
immobile.loc[known] = smoothed_speed.loc[known] <= IMMOBILITY_THRESHOLD_PX_S

clip = slice(clip_start, clip_stop)
t = (df.index.to_numpy()[clip] - clip_start) / FPS
plt.plot(t, selected_speed.iloc[clip], alpha=0.4, label="Raw speed")
plt.plot(t, smoothed_speed.iloc[clip], label="Rolling mean")
plt.axhline(IMMOBILITY_THRESHOLD_PX_S, color="black", linestyle="--", label="Illustrative threshold")
plt.xlabel("Time from clip start (s)")
plt.ylabel("Speed (pixels/s)")
plt.legend()
plt.show()
print(immobile.value_counts(dropna=False))


## Optional advanced: an event table has one row per bout

Frame metrics and events have different row meanings, so store them separately. Unknown samples split
candidate bouts; bouts touching unknown data or recording boundaries are marked censored. Their full
durations are not known. This example uses the selected animal only and imposes no minimum duration.


In [ ]:
is_rest = immobile.fillna(False).to_numpy(dtype=bool)
changes = np.diff(is_rest.astype(int), prepend=0, append=0)
starts = np.flatnonzero(changes == 1)
stops = np.flatnonzero(changes == -1)  # Exclusive.
unknown = immobile.isna().to_numpy()
bouts = pd.DataFrame({"individual": animal, "start_frame": starts, "stop_frame_exclusive": stops})
bouts["observed_duration_s"] = (stops - starts) / FPS
bouts["left_censored"] = [s == 0 or unknown[s - 1] for s in starts]
bouts["right_censored"] = [e == len(unknown) or unknown[e] for e in stops]
bouts.head(10)


## Optional: save results and analysis settings

Preserve frame numbers, units, and the parameters needed to understand the results. The export is opt-in.
Synthetic and experimental outputs have different filename prefixes.


In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    import json
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prefix = "demo" if USE_DEMO_DATA else DATA_FILE.stem
    metrics.to_csv(OUTPUT_DIR / f"{prefix}_frame_metrics.csv", index=True)
    summary.to_csv(OUTPUT_DIR / f"{prefix}_summary.csv", index=True)
    bouts.to_csv(OUTPUT_DIR / f"{prefix}_candidate_bouts.csv", index=False)
    settings = dict(synthetic=USE_DEMO_DATA, source=None if USE_DEMO_DATA else str(DATA_FILE),
                    fps=FPS, confidence_threshold=CONFIDENCE_THRESHOLD,
                    max_gap_frames=MAX_GAP_FRAMES, center_parts=CENTER_PARTS,
                    window_frames=WINDOW_FRAMES,
                    illustrative_immobility_threshold_px_s=IMMOBILITY_THRESHOLD_PX_S)
    (OUTPUT_DIR / f"{prefix}_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    print(f"Saved results to {OUTPUT_DIR.resolve()}")


## Presenter appendix: prepare matched video clips

Run this before the meeting with real data. This helper is not part of the pandas/NumPy lesson.
It writes the same exclusive frame range for all three stages, checks video/table alignment and frame rate,
and keeps video colors separate from plotting colors. Synthetic data are not aligned with a real video.


In [ ]:
def export_tracking_clip(table, video_path, output_path, start, stop, expected_fps):
    import cv2
    cap = cv2.VideoCapture(str(video_path))
    writer = None
    try:
        if not cap.isOpened():
            raise OSError(f"Cannot open {video_path}")
        video_fps = cap.get(cv2.CAP_PROP_FPS)
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if not np.isclose(video_fps, expected_fps, rtol=0.001):
            raise ValueError(f"Video FPS {video_fps} differs from configured FPS {expected_fps}")
        if n_frames != len(table):
            raise ValueError("Video and tracking lengths differ; check frame alignment first.")
        if not 0 <= start < stop <= n_frames:
            raise ValueError("Invalid clip range")
        size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
        writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"MJPG"), video_fps, size)
        if not writer.isOpened():
            raise OSError(f"Cannot create {output_path}")
        palette = [(255, 0, 255), (0, 255, 255), (255, 255, 0)]  # OpenCV BGR.
        individuals = table.columns.get_level_values("individuals").unique()
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        for frame_number in range(start, stop):
            ok, frame = cap.read()
            if not ok:
                raise OSError(f"Could not read frame {frame_number}")
            for i, individual in enumerate(individuals):
                parts = table[individual].columns.get_level_values("bodyparts").unique()
                for bodypart in parts:
                    x = table.iloc[frame_number][(individual, bodypart, "x")]
                    y = table.iloc[frame_number][(individual, bodypart, "y")]
                    if np.isfinite(x) and np.isfinite(y):
                        cv2.circle(frame, (int(x), int(y)), 5, palette[i % len(palette)], -1)
            writer.write(frame)
    finally:
        cap.release()
        if writer is not None:
            writer.release()
    print(f"Saved {output_path}")

EXPORT_VIDEOS = False
if EXPORT_VIDEOS:
    if USE_DEMO_DATA:
        raise ValueError("Choose real tracking data before exporting annotated videos.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for stage, table in [("original", df), ("filtered", filtered_df), ("interpolated", interpolated_df)]:
        export_tracking_clip(table, VIDEO_FILE, OUTPUT_DIR / f"{stage}_clip.avi",
                             clip_start, clip_stop, FPS)
